# Lab 13: many fine-tunes on one base

Two rank-8 adapters made in the notebook (random weights: this lab measures cost, not quality) behind one server, and what each costs per step.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Labs 08 and 10 are read-only on this GPU (no FP8 arithmetic).")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: peft, to write the adapters. Kaggle ships an old torchao that peft refuses, so it is removed.
%pip install -q "peft==0.21.2" 2>&1 | tail -1
%pip uninstall -y -q torchao

In [ ]:
# Cell 3: helpers used by every lab: start the server in the background, wait for it, talk to it.
import json, re, subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
URL = "http://127.0.0.1:8000"

def serve(extra_args=(), log_path="/kaggle/working/serve.log", timeout_s=900):
    args = ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", *extra_args]   # the T4 has no bfloat16
    log = open(log_path, "w")
    t0 = time.time()
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("vllm serve exited; see " + log_path)
        try:
            urllib.request.urlopen(URL + "/health", timeout=2); return proc, round(time.time() - t0, 1)
        except Exception:
            time.sleep(0.5)
    raise RuntimeError("server not ready in time")

def stop(proc):
    proc.terminate(); proc.wait()

def log_lines(path, patterns):
    text = open(path).read(); out = {}
    for name, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            line = re.sub(r"^\(EngineCore[^)]*\) ", "", m.group(0)); line = re.sub(r"^INFO \d\d-\d\d \d\d:\d\d:\d\d \[[^\]]+\] ", "", line); out[name] = line.strip()
    return out

STARTUP = {"dtype": r"INFO[^\n]*dtype=torch\.[a-z0-9]+", "backend": r"Using [A-Z_]+ attention backend[^\n]*", "weights": r"Loading weights took [^\n]*",
           "model": r"Model loading took [^\n]*", "kv_memory": r"Available KV cache memory: [^\n]*", "kv_size": r"GPU KV cache size: [^\n]*", "graphs": r"Graph capturing finished[^\n]*"}

def chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())
print("helpers ready")

In [ ]:
def stream_chat(messages, **params):
    body = json.dumps({"model": params.pop("model", MODEL), "messages": messages, "stream": True, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    t0 = time.time(); first = None; last = None; gaps = []; text = []
    with urllib.request.urlopen(req, timeout=600) as resp:
        for raw in resp:
            line = raw.decode().strip()
            if not line.startswith("data:") or line == "data: [DONE]": continue
            delta = json.loads(line[5:])["choices"][0].get("delta", {}).get("content")
            if not delta: continue
            now = time.time()
            if first is None: first = now - t0
            else: gaps.append(now - last)
            last = now; text.append(delta)
    return first, gaps, "".join(text)

def metrics(names):
    out = {}
    for line in urllib.request.urlopen(URL + "/metrics", timeout=5).read().decode().splitlines():
        if line.startswith("#"): continue
        for n in names:
            if line.startswith(n + "{") or line.startswith(n + " "):
                try: out[n] = float(line.rsplit(" ", 1)[1])
                except ValueError: pass
    return out
print("streaming and metrics helpers ready")

In [ ]:
# Cell 6: the load generator and the benchmark call.
import random, threading
def random_prompt(n_tokens, seed):
    rng = random.Random(seed); words = ["river", "stone", "quiet", "engine", "orange", "ledger", "window", "copper", "signal", "harbor", "meadow", "lantern", "velvet", "summit", "cinder", "fable"]
    return " ".join(rng.choice(words) for _ in range(int(n_tokens * 0.8)))

def load_test(rate, num_prompts, prompt_len=512, out_len=128, prefix_len=0, model=None, concurrency=None):
    rng = random.Random(1); prefix = random_prompt(prefix_len, 99) + " " if prefix_len else ""
    prompts = [prefix + random_prompt(prompt_len - prefix_len, i) for i in range(num_prompts)]; results = []; lock = threading.Lock()
    def one(i):
        first, gaps, text = stream_chat([{"role": "user", "content": prompts[i]}], **({"model": model} if model else {}), max_tokens=out_len, temperature=0, ignore_eos=True)
        with lock: results.append((first, (sum(gaps) / len(gaps)) if gaps else 0.0, len(gaps) + 1))
    sem = threading.Semaphore(concurrency or num_prompts); t0 = time.time(); threads = []
    def gated(i):
        with sem: one(i)
    for i in range(num_prompts):
        th = threading.Thread(target=gated, args=(i,)); th.start(); threads.append(th)
        if not concurrency: time.sleep(rng.expovariate(rate))
    for th in threads: th.join()
    dur = time.time() - t0; ttft = sorted(r[0] for r in results); tpot = sorted(r[1] for r in results); q = lambda xs, p: xs[min(len(xs) - 1, int(p * len(xs)))]
    return {"req_s": round(len(results) / dur, 2), "out_tok_s": round(sum(r[2] for r in results) / dur, 1), "ttft_p50_ms": round(1000 * q(ttft, 0.5)), "ttft_p99_ms": round(1000 * q(ttft, 0.99)), "tpot_p50_ms": round(1000 * q(tpot, 0.5), 1), "tpot_p99_ms": round(1000 * q(tpot, 0.99), 1)}

def load_row(label, r):
    return f"{label:<30} req/s {r['req_s']:5.2f}  out tok/s {r['out_tok_s']:7.1f}  TTFT p50 {r['ttft_p50_ms']:6d} p99 {r['ttft_p99_ms']:6d} ms  TPOT p50 {r['tpot_p50_ms']:5.1f} p99 {r['tpot_p99_ms']:5.1f} ms"
print("load generator ready")
def bench_quiet(rate, num_prompts, input_len=1024, output_len=128, max_concurrency=None, model=None):
    args = ["vllm", "bench", "serve", "--model", model or MODEL, "--tokenizer", MODEL, "--port", "8000", "--dataset-name", "random", "--random-input-len", str(input_len),
            "--random-output-len", str(output_len), "--num-prompts", str(num_prompts), "--request-rate", str(rate),
            "--metric-percentiles", "50,95,99", "--percentile-metrics", "ttft,tpot,itl,e2el", "--ignore-eos", "--seed", "0"]
    if max_concurrency: args += ["--max-concurrency", str(max_concurrency)]
    out = subprocess.run(args, capture_output=True, text=True); text = out.stdout; i = text.find("Serving Benchmark Result")
    res = {}
    for line in text.splitlines():
        m = re.match(r'^(.*?):\s+([0-9.]+)\s*$', line)
        if m: res[m.group(1).strip()] = float(m.group(2))
    return res
def row(label, r):
    g = lambda k: r.get(k, float("nan"))
    return f"{label:<28} req/s {g('Request throughput (req/s)'):5.2f}  out tok/s {g('Output token throughput (tok/s)'):7.1f}  TTFT p50 {g('P50 TTFT (ms)'):6.0f} p99 {g('P99 TTFT (ms)'):6.0f} ms  TPOT p50 {g('P50 TPOT (ms)'):5.1f} p99 {g('P99 TPOT (ms)'):5.1f} ms  peak conc {g('Maximum request concurrency'):3.0f}"


In [ ]:
# Cell 7: two adapters.
import os, torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16)
for name in ("legal", "support"):
    m = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_dropout=0.0))
    for p in m.parameters():
        if p.requires_grad: torch.nn.init.normal_(p, std=0.02)
    m.save_pretrained(f"/kaggle/working/lora-{name}"); del m
    print(f"  adapter {name}: {sum(os.path.getsize(os.path.join(f'/kaggle/working/lora-{name}', f)) for f in os.listdir(f'/kaggle/working/lora-{name}')) / 1e6:.1f} MB")
del base; torch.cuda.empty_cache()

In [ ]:
# Cell 8: one server, the base plus both adapters.
proc, ready_s = serve(["--max-model-len", "4096", "--enable-lora", "--max-lora-rank", "8", "--lora-modules", "legal=/kaggle/working/lora-legal", "support=/kaggle/working/lora-support"]); print(f"ready after {ready_s} s")
for label, model in (("base", MODEL), ("adapter legal", "legal"), ("adapter support", "support")):
    print(row(f"{label}, 1 at a time", bench_quiet(rate="inf", num_prompts=16, input_len=512, max_concurrency=1, model=model)))
print(row("base x32", bench_quiet(rate="inf", num_prompts=64, input_len=512, max_concurrency=32, model=MODEL)))
res = {}
def go(name): res[name] = load_test(rate=0, num_prompts=32, prompt_len=512, out_len=128, model=name, concurrency=16)
ts = [threading.Thread(target=go, args=(n,)) for n in ("legal", "support")]; [t.start() for t in ts]; [t.join() for t in ts]
print(f"mixed x32 (16 legal + 16 support)      out tok/s {sum(r['out_tok_s'] for r in res.values()):7.1f}  TPOT p50 {res['legal']['tpot_p50_ms']:5.1f} / {res['support']['tpot_p50_ms']:5.1f} ms")
stop(proc)